# UNIVERSAL ADAPTIVE PIPELINE: BINARY DETECTION & CLASS IMBALANCE
### COMPETITION-GRADE ARCHITECTURE (HOAKS, CYBERBULLYING, TOXIC COMMENTS, FRAUD DETECTION)

**Prinsip Desain:**
1. **Zero Hardcoded Lexicon**: Tidak mengandalkan daftar kata makian/hoaks kaku. Menggunakan kombinasi fitur stilometri struktural dan Char-WB N-grams yang secara matematis menangkap kata samaran (*leet speak*, sensor angka `4nj1ng`, tanda baca berulang) di semua bahasa.
2. **Auto-Discovery**: Menangani otomatis kolom ID, Teks, dan Biner Target (0 vs 1).
3. **Imbalance-Aware Models**: LightGBM (`scale_pos_weight`) dan Logistic Regression (`class_weight='balanced'`).
4. **Optimal Precision-Recall Threshold Sweep**: Mencari titik potong probabilitas terbaik (\(T \in [0.1, 0.9]\)) yang memaksimalkan F1-Score pada OOF, menggantikan asumsi kaku threshold 0.50.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Environment Binary Detection siap. Seed:", SEED)


## 1. Unified Configuration Card


In [ ]:
CONFIG = {
    "train_path": "cyberbullying_hoax_train.csv",
    "test_path": "cyberbullying_hoax_test.csv", # Isi None jika tidak ada test set
    
    "id_col": None,
    "text_cols": None,
    "target_col": None,
    
    # Preprocessing & Stylometrics
    "lower": True,
    "max_features": 15000,
    "n_splits": 5
}

print("Konfigurasi tersimpan.")


## 2. Auto-Discovery & Data Ingestion Engine


In [ ]:
def load_binary_data(cfg):
    tr_p, te_p = cfg["train_path"], cfg["test_path"]
    if not os.path.exists(tr_p):
        print(f"Peringatan: Berkas '{tr_p}' belum ada. Menjalankan fallback generator biner...")
        train_df = pd.DataFrame({
            "sample_id": range(1, 15),
            "text_body": [
                "Waspada! Beredar kabar telur palsu terbuat dari plastik beredar di pasar tradisional!",
                "Kementerian Kesehatan membantah isu vaksin menyebabkan chip pelacak.",
                "Dasar manusia gak punya otak kerjaannya cuma nyusahin orang lain doang!!",
                "Muka lu jelek banget kayak sampah gak pantas ikut kompetisi bergengsi ini wkwk",
                "Rapat koordinasi tingkat menteri menyepakati penurunan tarif impor pangan.",
                "BREAKING NEWS!! Minum air rebusan bawang putih bisa sembuhkan penyakit kronis 100%!",
                "Selamat kepada seluruh peserta yang telah berhasil lolos ke babak final nasional.",
                "Halah pansos lu artis modal tampang doang tapi kelakuan kayak binatang baji*gan",
                "Prakiraan cuaca wilayah Jawa Barat berpotensi cerah berawan hingga sore hari.",
                "KLAIM MENYESATKAN: Beredar video jembatan ambruk yang diklaim berlokasi di Jakarta.",
                "Mampus lu kena karma rasain akibatnya orang sombong sok jagoan!!",
                "Pemerintah resmi meresmikan jalan tol baru sepanjang 50 kilometer.",
                "Bocah tolol main game terus mending mati aja lu gak ada gunanya hidup!!",
                "Universitas menyelenggarakan workshop penulisan karya ilmiah bereputasi internasional."
            ],
            "target_label": [1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0] # 1 = Toxic/Hoax, 0 = Normal
        })
        test_df = None
    else:
        ext = os.path.splitext(tr_p)[1].lower()
        if ext in ['.xlsx', '.xls']: train_df = pd.read_excel(tr_p); test_df = pd.read_excel(te_p) if te_p and os.path.exists(te_p) else None
        elif ext == '.parquet': train_df = pd.read_parquet(tr_p); test_df = pd.read_parquet(te_p) if te_p and os.path.exists(te_p) else None
        else: train_df = pd.read_csv(tr_p); test_df = pd.read_csv(te_p) if te_p and os.path.exists(te_p) else None

    # Auto ID
    id_col = cfg["id_col"]
    if not id_col or id_col not in train_df.columns:
        id_cands = [c for c in train_df.columns if re.search(r"id|uuid|kode|no", c, re.IGNORECASE)]
        id_col = id_cands[0] if id_cands else train_df.columns[0]

    # Auto Target
    target_col = cfg["target_col"]
    if not target_col or target_col not in train_df.columns:
        target_cands = [c for c in train_df.columns if c != id_col and train_df[c].nunique() == 2]
        target_col = target_cands[0] if target_cands else train_df.columns[-1]

    # Auto Text
    text_cols = cfg["text_cols"]
    if not text_cols:
        text_cols = [c for c in train_df.columns if c not in [id_col, target_col] and (train_df[c].dtype == object or pd.api.types.is_string_dtype(train_df[c]))]
        if not text_cols: text_cols = [train_df.columns[1]]
    elif isinstance(text_cols, str):
        text_cols = [text_cols]

    return train_df, test_df, id_col, target_col, text_cols

train_df, test_df, ID_COL, TARGET_COL, TEXT_COLS = load_binary_data(CONFIG)
train_df = train_df.dropna(subset=[TARGET_COL] + TEXT_COLS).reset_index(drop=True)

# Pastikan label biner 0 dan 1
unique_targets = sorted(train_df[TARGET_COL].unique())
if set(unique_targets) != {0, 1}:
    t_map = {unique_targets[0]: 0, unique_targets[1]: 1}
    train_df["binary_target"] = train_df[TARGET_COL].map(t_map)
else:
    train_df["binary_target"] = train_df[TARGET_COL].astype(int)

pos_count = (train_df["binary_target"] == 1).sum()
neg_count = (train_df["binary_target"] == 0).sum()
imbalance_ratio = neg_count / max(1, pos_count)

print(f"=== SKEMA DETEKSI BINER TERIDENTIFIKASI ===")
print(f"ID Col          : '{ID_COL}'")
print(f"Target Col      : '{TARGET_COL}' (Positif: {pos_count}, Negatif: {neg_count})")
print(f"Imbalance Ratio : 1 : {imbalance_ratio:.2f}")
print(f"Text Col(s)     : {TEXT_COLS}")


## 3. Ekstraksi Fitur: Morfometri Struktural + Char-WB N-Grams
Char-wb menangkap leet-speak / sensor huruf (contoh: `4nj1ng`, `g0bl0k`) secara matematis tanpa kamus.


In [ ]:
# 1. Fitur Struktural Universal
def extract_universal_binary_features(df, text_series):
    feats = pd.DataFrame(index=df.index)
    s = text_series.astype(str)
    feats["char_len"] = s.str.len()
    feats["word_count"] = s.apply(lambda x: len(x.split()))
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["caps_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isupper())) / (feats["char_len"] + 1)
    feats["exclam_count"] = s.apply(lambda x: x.count('!'))
    feats["question_count"] = s.apply(lambda x: x.count('?'))
    feats["punct_count"] = s.apply(lambda x: len(re.findall(r"[^\w\s]", x)))
    feats["repeated_char_seq"] = s.apply(lambda x: len(re.findall(r"(.)\1{2,}", x)))
    feats["digit_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isdigit())) / (feats["char_len"] + 1)
    return feats

# Gabungkan teks jika >1 kolom
def fuse_text(df, cols):
    series = [df[c].fillna("").astype(str) for c in cols]
    fused = series[0]
    for s in series[1:]: fused = fused + " " + s
    return fused.str.strip()

train_df["fused_text"] = fuse_text(train_df, TEXT_COLS)
if test_df is not None: test_df["fused_text"] = fuse_text(test_df, TEXT_COLS)

style_tr = extract_universal_binary_features(train_df, train_df["fused_text"])
style_te = extract_universal_binary_features(test_df, test_df["fused_text"]) if test_df is not None else None

# TF-IDF Word (1, 2) + Char-WB (3, 5)
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=CONFIG["max_features"], sublinear_tf=True)
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer="char_wb", max_features=CONFIG["max_features"], sublinear_tf=True)

X_tr_w = word_vec.fit_transform(train_df["fused_text"].str.lower())
X_tr_c = char_vec.fit_transform(train_df["fused_text"].str.lower())
X_train = hstack([X_tr_w, X_tr_c, csr_matrix(style_tr.values)]).tocsr()

if test_df is not None:
    X_te_w = word_vec.transform(test_df["fused_text"].str.lower())
    X_te_c = char_vec.transform(test_df["fused_text"].str.lower())
    X_test = hstack([X_te_w, X_te_c, csr_matrix(style_te.values)]).tocsr()
else:
    X_test = None

print(f"Total Dimensi Fitur Gabungan: {X_train.shape}")


## 4. Validasi Silang 5-Fold & Optimasi Ambang Batas (Threshold Sweep)


In [ ]:
y_train = train_df["binary_target"].values
n_splits = min(CONFIG["n_splits"], min(pos_count, neg_count))
n_splits = max(2, n_splits)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

oof_probas = np.zeros(len(train_df))
test_probas = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]
    
    # Logistic Regression dengan pembobotan kelas terbalik
    clf = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED + fold)
    clf.fit(X_tr, y_tr)
    
    probas = clf.predict_proba(X_va)[:, 1]
    oof_probas[va_idx] = probas
    if X_test is not None: test_probas.append(clf.predict_proba(X_test)[:, 1])

# Precision-Recall Threshold Sweep (Mencari T optimal yang memaksimalkan F1)
thresholds = np.linspace(0.1, 0.9, 81)
best_t, best_f1 = 0.5, 0.0

for t in thresholds:
    f1 = f1_score(y_train, (oof_probas >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

default_f1 = f1_score(y_train, (oof_probas >= 0.5).astype(int), zero_division=0)
auc_score = roc_auc_score(y_train, oof_probas)

print(f"\n=== HASIL EVALUASI OUT-OF-FOLD (OOF) ===")
print(f"ROC-AUC Score          : {auc_score:.4f}")
print(f"Default Threshold 0.50 : F1 = {default_f1:.4f}")
print(f"Optimal Threshold {best_t:.2f} : F1 = {best_f1:.4f} (+{(best_f1 - default_f1)*100:.2f}%)")
print(f"Precision @ {best_t:.2f}   : {precision_score(y_train, (oof_probas >= best_t).astype(int), zero_division=0):.4f}")
print(f"Recall @ {best_t:.2f}      : {recall_score(y_train, (oof_probas >= best_t).astype(int), zero_division=0):.4f}")


## 5. Ekspor Berkas Submission Format Biner & Probabilitas


In [ ]:
if test_df is not None and len(test_probas) > 0:
    avg_test_proba = np.mean(test_probas, axis=0)
    final_binary = (avg_test_proba >= best_t).astype(int)
    
    sub = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        "predicted_label": final_binary,
        "probability": np.round(avg_test_proba, 4)
    })
    sub.to_csv("submission_binary_detection.csv", index=False)
    print("Berkas submission_binary_detection.csv berhasil disimpan.")
    print(sub.head())
